# Biblical RAG LLM MVP 1

This notebook tests llama3's ability to read a Bible that is recursively chunked.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from uuid import uuid4
import json
import os

# Open King James Bible.
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')
kjv_df = pd.read_csv(os.path.join(data_dir, 'en_kjv.csv'), index_col='index')
kjv_df.head()

,language,translation,book,chapter,verse,text
index,,,,,,
0,en,kjv,Gen,1,1,In the beginning God created the heaven and th...
1,en,kjv,Gen,1,2,"And the earth was without form, and void; and ..."
2,en,kjv,Gen,1,3,"And God said, Let there be light: and there wa..."
3,en,kjv,Gen,1,4,"And God saw the light, that it was good: and G..."
4,en,kjv,Gen,1,5,"And God called the light Day, and the darkness..."


In [2]:
# Open metadata table.
book_metadata = pd.read_csv(os.path.join(data_dir, 'metadata', 'books.csv'))
book_metadata.loc[book_metadata['book'].isin(['Numbers', 'Jonah', 'Ruth', 'Mark', 'Titus', 'Revelation'])]

,book,chapters,verses,avg verse per chapter,testament,category,author,abbreviation
3,Numbers,36,1288,36,old,law,Moses,Num
7,Ruth,4,85,21,old,writing,Samuel,Ruth
31,Jonah,4,48,12,old,prophet,Jonah,Jonah
40,Mark,16,678,42,new,gospel,Mark,Mark
55,Titus,3,46,15,new,epistle,Paul,Titus
65,Revelation,22,404,18,new,revelation,John,Rev


## Prepare the Data

In [3]:
# Combine all verses into books.
kjv_books = {}
book_title_list = book_metadata['abbreviation'].values
tween = ' ' # What goes between each verse
verse_total = book_metadata['verses'].sum() # Index of the final verse
print("Begin forming books...")
for book in book_title_list:
    book_df = kjv_df.loc[kjv_df['book'] == book]
    full_title = book_metadata.loc[book_metadata['abbreviation'] == book]['book'].values[0]
    full_book_text = ''
    for index, row in book_df.iterrows():
        full_book_text += row['text'] + tween
    kjv_books[full_title] = full_book_text
print("Finished forming books!")
print(kjv_books.keys())
# Store in file
out_path = os.path.join(data_dir, 'en_kjv.json')
if not os.path.isfile(out_path):
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(kjv_books, f, indent=4, ensure_ascii=False)

Begin forming books...
Finished forming books!
dict_keys(['Genesis', 'Exodus', 'Leviticus', 'Numbers', 'Deuteronomy', 'Joshua', 'Judges', 'Ruth', '1 Samuel', '2 Samuel', '1 Kings', '2 Kings', '1 Chronicles', '2 Chronicles', 'Ezra', 'Nehemiah', 'Esther', 'Job', 'Psalms', 'Proverbs', 'Ecclesiastes', 'Song of Songs', 'Isaiah', 'Jeremiah', 'Lamentations', 'Ezekiel', 'Daniel', 'Hosea', 'Joel', 'Amos', 'Obadiah', 'Jonah', 'Micah', 'Nahum', 'Habakkuk', 'Zephaniah', 'Haggai', 'Zechariah', 'Malachi', 'Matthew', 'Mark', 'Luke', 'John', 'Acts', 'Romans', '1 Corinthians', '2 Corinthians', 'Galatians', 'Ephesians', 'Philippians', 'Colossians', '1 Thessalonians', '2 Thessalonians', '1 Timothy', '2 Timothy', 'Titus', 'Philemon', 'Hebrews', 'James', '1 Peter', '2 Peter', '1 John', '2 John', '3 John', 'Jude', 'Revelation'])


In [4]:
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

local_llm = 'llama3'
token_size=500
dimensions=300

# Convert each book into a document.
def document_verses(doc_dict):
    documents = []
    count = 0 # Count batches for sanity's sake.
    for doc_index, doc in enumerate(doc_dict):
        count += 1
        doc = Document(
            page_content=doc_dict[doc],
            metadata={
                'title': doc,
                'author': book_metadata.loc[book_metadata['book'] == doc]['author'].values[0],
                'book_index': doc_index
            }, ################## TO DO: IMPORT THE CHAPTER_VERSE_METADATA FUNCTION ##################################################################
            id=count
        )
        documents.append(doc)
    uuids = [str(uuid4()) for _ in range(len(documents))]
    return documents, uuids

# Split the documents.
kjv_docs, kjv_ids = document_verses(kjv_books)
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=token_size, chunk_overlap=100
)
split_docs = text_splitter.split_documents(kjv_docs)
print("Documents split.")

Documents split.


In [19]:
# Check the documents that have been split
print("There are {} chunks.\n".format(len(split_docs)))
for i in range(8):
    r = split_docs[i*300]
    print("{} ({}, by {})".format(r.page_content, r.metadata['title'], r.metadata['author']))###### TO DO: CHAPTER AND VERRSE REFS ####
    print('')

There are 2498 chunks.

In the beginning God created the heaven and the earth. And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters. And God said, Let there be light: and there was light. And God saw the light, that it was good: and God divided the light from the darkness. And God called the light Day, and the darkness he called Night. And the evening and the morning were the first day. And God said, Let there be a firmament in the midst of the waters, and let it divide the waters from the waters. And God made the firmament, and divided the waters which were under the firmament from the waters which were above the firmament: and it was so. And God called the firmament Heaven. And the evening and the morning were the second day. And God said, Let the waters under the heaven be gathered together unto one place, and let the dry land appear: and it was so. And God called the dry land Earth; and the gat

## Store the Data

In [5]:
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_community.vectorstores import FAISS

print("Begin instantiating LLM.")
llm = OllamaLLM(model='llama3')
print("LLM instantiated.")
embedding = OllamaEmbeddings(model='llama3')
vectorstore = FAISS.from_documents(split_docs, embedding)
print("Vector store instantiated.")

LLM instantiated.
Vector store instantiated.


### Save & Load

In [ ]:
vector_file = os.path.join(data_dir, 'bible_csv.db')
vectorstore.save_local(vector_file)
print("File saved.")

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embedding = OllamaEmbeddings(model='llama3')
vector_file = os.path.join(data_dir, 'bible_csv.db')
vectorstore = FAISS.load_local(vector_file, embedding, allow_dangerous_deserialization=True)#.as_retriever() if you want to load it as a retriever
print("File loaded.")

## RAG Chain

In [7]:
from langchain.chains import RetrievalQA

print("Begin instantiating retriever.")
retriever = vectorstore.as_retriever() ################## TO DO: FIND OUT IF THIS NEEDS TO BE ALTERED TO SHOW MORE TEXTS ############
print("Retriever instantiated.")
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", ################## TO DO: FIND OUT WHAT CHAIN_TYPE IS SUPPOSED TO BE ###########################
    retriever=retriever
)
print("RAG chain instantiated.")

Begin instantiating retriever.
Retriever instantiated.
RAG chain instantiated.


## Test the RAG Model

In [22]:
questions = [
    "Where did Jesus talk about wine?",
    "Who was Abraham's son?",
    "Where did God send Jonah? And where did Jonah try to go instead?",
    "Who are Luke's books addressed to?",
    "Which psalm says that angels will protect you even from dashing your foot against a stone?",
    "Why did God regret making Saul the King of Israel?",
    "What did Paul disagree with Peter about according to Paul's letter to the Galatians? Who was right?",
    "Why did Peter feel comfortable eating pork and shellfish after receiving the Holy Spirit?",
    "When Jesus mentions that Moses lifted the serpent in the wilderness (John 3:14), what is He referencing?",
    "What is the genealogy of Enoch going back to Adam?",
    "Why did God flood the world in the time of Noah?",
    "What does Messiah mean in Hebrew?",
    "How do you spell 'Jesus' in Hebrew and Greek?",
    "What are the similarities between Daniel's prophesies and Revelation's prophesies?"
]
print("There are {} questions in the test suite.".format(len(questions)))

for question in questions:
    answer = qa.invoke(question)
    print(answer)

There are 14 questions in the test suite.
{'query': 'Where did Jesus talk about wine?', 'result': 'There is no information provided in the given context about Jesus talking about wine. The text appears to be a collection of biblical passages from Exodus, describing events such as the plague of locusts and the tenth plague of Egypt. There is no mention of Jesus or any discussion about wine.'}
{'query': "Who was Abraham's son?", 'result': "According to the provided context, there is no specific mention of Abraham or his sons. The texts seem to be from different sources, including biblical accounts (e.g., Genesis 12-25) and possibly other ancient writings.\n\nHowever, based on biblical records, Abraham had two significant sons:\n\n1. Isaac: He was Abraham's son with Sarah (Genesis 21).\n2. Ishmael: He was Abraham's son with Hagar (Genesis 16).\n\nPlease clarify or provide more context if you're looking for a specific reference or connection to one of these individuals."}
{'query': 'Where 

## Conclusion

### Where did Jesus talk about wine?
The model does not answer the question.
### Who was Abraham's son?
The model correctly identifies Abraham's sons, but not from the RAG texts.
### Where did God send Jonah? And where did Jonah try to go instead?
The model does not answer the question.
### Who are Luke's books addressed to?
The model does not answer the question.
### Which psalm says that angels will protect you even from dashing your foot against a stone?
The model does not answer the question.
### Why did God regret making Saul the King of Israel?
The model does not answer the question.
### What did Paul disagree with Peter about according to Paul's letter to the Galatians? Who was right?
The model correctly identifies circumcision as the source of disagreement, but mentions that it did not use the RAG texts. This may have been easier to answer because a specific book is mentioned in the question.
### Why did Peter feel comfortable eating pork and shellfish after receiving the Holy Spirit?
The model is confused by the RAG texts being given to it. It is not able to give the correct answer because of this confusion. **I believe the RAG model is not providing relevant texts.**
### When Jesus mentions that Moses lifted the serpent in the wilderness (John 3:14), what is He referencing?
The model correctly identifies that the reference comes from the Book of Numbers, and its meaning. Again, this might be because a specific passage is mentioned in the question.
### What is the genealogy of Enoch going back to Adam?
The model correctly identifies the geneology, while also denying that such a geneology exists. This hallucination is not likely to be RAG-attributed.
### Why did God flood the world in the time of Noah?
The model correctly identifies the reason for the flood, and tells the story accurately, giving correct references to Genesis.
### What does Messiah mean in Hebrew?
The model gives the correct answer, but it also says that the RAG texts do not mention the word 'Messiah'. **I wonder what the RAG algorithm is doing?**
### How do you spell 'Jesus' in Hebrew and Greek?
The model gives a focused answer that spells out the name in both Greek and Hebrew letters, but only shows Greek letters.
### What are the similarities between Daniel's prophesies and Revelation's prophesies?
The model doesn't seem to know what Daniel and Revelation are, even though a non-RAG model would know what they are.
### Score:
7/12